# AI Candidate Ranking System
**Run this on GPU runtime for fastest results.**
Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 1: Install dependencies
!pip install sentence-transformers faiss-gpu tqdm -q

In [ ]:
# Step 2: Upload your files
from google.colab import files
import os

print('Upload candidates.jsonl')
uploaded = files.upload()  # upload candidates.jsonl

print('Upload job_description.txt')
uploaded2 = files.upload()  # upload job_description.txt

In [ ]:
# Step 3: Full pipeline
import json, csv, os
from tqdm import tqdm
from datetime import date, datetime
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# ── Config ────────────────────────────────────────────────────────
CANDIDATES_FILE = 'candidates.jsonl'
JD_FILE = 'job_description.txt'
OUTPUT_FILE = 'submission.csv'
TOP_N = 100
POOL = 500

# ── Load data ─────────────────────────────────────────────────────
candidates = []
with open(CANDIDATES_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            candidates.append(json.loads(line))
print(f'Loaded {len(candidates):,} candidates')

with open(JD_FILE, 'r') as f:
    jd = f.read()
print(f'JD loaded ({len(jd)} chars)')

candidate_map = {c['candidate_id']: c for c in candidates}

In [ ]:
# ── Embedding ──────────────────────────────────────────────────────
def build_text(c):
    parts = []
    p = c.get('profile', {})
    parts.append(f"{p.get('current_title','')} at {p.get('current_company','')} {p.get('years_of_experience',0)}yr")
    if p.get('headline'): parts.append(p['headline'][:100])
    skills = c.get('skills', [])
    top_skills = [s['name'] for s in skills if s.get('proficiency') in ('expert','advanced')][:8]
    if top_skills: parts.append(f"Skills: {', '.join(top_skills)}")
    for role in c.get('career_history', [])[:2]:
        parts.append(f"{role.get('title','')}: {role.get('description','')[:150]}")
    return ' | '.join(filter(None, parts))

print('Loading embedding model...')
model = SentenceTransformer('all-MiniLM-L6-v2')

print('Embedding candidates (GPU accelerated)...')
texts = [build_text(c) for c in candidates]
vectors = model.encode(texts, batch_size=512, show_progress_bar=True,
                       normalize_embeddings=True, convert_to_numpy=True,
                       device='cuda')
print(f'Vectors shape: {vectors.shape}')

In [ ]:
# ── FAISS Index ────────────────────────────────────────────────────
dim = vectors.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(vectors.astype(np.float32))
candidate_ids = [c['candidate_id'] for c in candidates]
print(f'FAISS index built: {index.ntotal:,} vectors')

# Embed JD and search
jd_vec = model.encode([jd], normalize_embeddings=True, convert_to_numpy=True, device='cuda')[0]
scores, indices = index.search(np.array([jd_vec], dtype=np.float32), POOL)
top_results = [(candidate_ids[idx], float(score)) for score, idx in zip(scores[0], indices[0]) if idx >= 0]
print(f'Semantic search done: top {len(top_results)} retrieved')

In [ ]:
# ── Trajectory Analysis ────────────────────────────────────────────
AI_ML_TITLES = {'ai engineer','ml engineer','machine learning engineer','nlp engineer',
    'applied scientist','research scientist','data scientist','llm engineer',
    'senior ai engineer','senior ml engineer','lead ai engineer','applied ml engineer',
    'computer vision engineer','deep learning engineer','staff ml engineer'}

CONSULTING = {'tcs','infosys','wipro','accenture','cognizant','hcl',
              'tech mahindra','capgemini','mphasis','hexaware','ltimindtree'}

MGMT_NONTECT = {'engineering manager','vp engineering','product manager','marketing manager',
    'operations manager','hr manager','sales manager','business analyst','project manager'}

def classify(title):
    t = title.lower()
    if any(x in t for x in AI_ML_TITLES): return 'ai_ml'
    if any(x in t for x in MGMT_NONTECT): return 'non_tech'
    return 'other'

def get_seniority(title):
    t = title.lower()
    for kw, lvl in [('principal',5),('staff',4),('lead',4),('senior',3),('junior',1),('intern',0)]:
        if kw in t: return lvl
    return 2

def trajectory(c):
    career = list(reversed(c.get('career_history', [])))
    if not career: return {'score': 0.2, 'pattern': 'NO_HISTORY', 'ai_focus_ratio': 0}
    total = sum(r.get('duration_months',0) for r in career) or 1
    ai_months = sum(r.get('duration_months',0) for r in career if classify(r.get('title',''))=='ai_ml')
    cons_months = sum(r.get('duration_months',0) for r in career if any(x in r.get('company','').lower() for x in CONSULTING))
    ai_ratio = ai_months / total
    cons_ratio = cons_months / total
    cats = [classify(r.get('title','')) for r in career]
    recent = cats[-1] if cats else 'other'
    senl = [get_seniority(r.get('title','')) for r in career]
    momentum = min(max(((sum(senl[-2:])/max(len(senl[-2:]),1)) - (sum(senl[:2])/max(len(senl[:2]),1)) + 3) / 6, 0), 1) if len(senl)>=2 else 0.5

    if cons_ratio > 0.85:
        return {'score': 0.15, 'pattern': 'CONSULTING_ONLY', 'ai_focus_ratio': ai_ratio}
    if recent == 'non_tech':
        return {'score': 0.20, 'pattern': 'PIVOTING_OUT', 'ai_focus_ratio': ai_ratio}
    if all(x=='ai_ml' for x in cats) and momentum >= 0.5:
        return {'score': min(1.0 + momentum*0.1, 1.0), 'pattern': 'FOCUSED_GROWTH', 'ai_focus_ratio': ai_ratio}
    if all(x=='ai_ml' for x in cats):
        return {'score': 0.75, 'pattern': 'STEADY_CONTRIBUTOR', 'ai_focus_ratio': ai_ratio}
    if recent == 'ai_ml' and ai_ratio >= 0.3:
        return {'score': 0.70, 'pattern': 'PIVOTING_IN', 'ai_focus_ratio': ai_ratio}
    if ai_ratio >= 0.5:
        return {'score': 0.60, 'pattern': 'MIXED_AI_FOCUS', 'ai_focus_ratio': ai_ratio}
    return {'score': max(0.10, ai_ratio), 'pattern': 'ADJACENT', 'ai_focus_ratio': ai_ratio}

print('Trajectory module ready')

In [ ]:
# ── Scoring ────────────────────────────────────────────────────────
TIER1 = {'embeddings','vector search','faiss','pinecone','rag','semantic search',
         'llm','transformers','nlp','pytorch','fine-tuning','bert','huggingface',
         'ranking','re-ranking','information retrieval','sentence-transformers'}
PROF_W = {'expert':1.0,'advanced':0.8,'intermediate':0.5,'beginner':0.2}

def skill_score(c):
    score = 0.0
    for s in c.get('skills',[]):
        name = s.get('name','').lower()
        prof = PROF_W.get(s.get('proficiency','beginner'), 0.2)
        end = min(s.get('endorsements',0),100)/100
        trust = prof * (0.6 + 0.4*end)
        if any(t in name for t in TIER1): score += trust
    # Assessment bonus
    for sk, val in c.get('redrob_signals',{}).get('skill_assessment_scores',{}).items():
        if any(t in sk.lower() for t in TIER1): score += (val/100)*0.15
    return min(score/3.0, 1.0)

def behavioral_score(c):
    sig = c.get('redrob_signals',{})
    score = 0.0
    if sig.get('open_to_work_flag'): score += 0.25
    notice = sig.get('notice_period_days',90)
    score += 0.20 if notice==0 else 0.15 if notice<=30 else 0.08 if notice<=60 else 0
    score += sig.get('recruiter_response_rate',0)*0.20
    la = sig.get('last_active_date','')
    if la:
        try:
            days = (date.today() - datetime.strptime(la,'%Y-%m-%d').date()).days
            score += 0.15 if days<=7 else 0.10 if days<=30 else 0.05 if days<=90 else 0
        except: pass
    score += sig.get('interview_completion_rate',0)*0.10
    score += min(sig.get('github_activity_score',0)/100,1.0)*0.10
    return min(score, 1.0)

def edu_bonus(c):
    for e in c.get('education',[]):
        t = e.get('tier','')
        if t=='tier_1': return 0.08
        if t=='tier_2': return 0.04
    return 0.0

def compute_penalty(c, traj):
    pen = 0.0
    if traj['pattern'] == 'CONSULTING_ONLY': pen += 0.40
    if traj['pattern'] == 'PIVOTING_OUT': pen += 0.35
    sig = c.get('redrob_signals',{})
    la = sig.get('last_active_date','')
    if la:
        try:
            days = (date.today() - datetime.strptime(la,'%Y-%m-%d').date()).days
            if days > 180 and sig.get('recruiter_response_rate',1) < 0.1: pen += 0.30
        except: pass
    short = [r for r in c.get('career_history',[]) if r.get('duration_months',12)<12 and not r.get('is_current')]
    if len(short) >= 3: pen += 0.15
    return min(pen, 0.6)

def hybrid_score(sem, c):
    traj = trajectory(c)
    ss = skill_score(c)
    bs = behavioral_score(c)
    eb = edu_bonus(c)
    pen = compute_penalty(c, traj)
    raw = 0.35*sem + 0.30*traj['score'] + 0.20*ss + 0.15*bs + eb
    return {
        'final_score': round(max(raw-pen, 0), 4),
        'semantic_score': round(sem, 4),
        'trajectory_score': round(traj['score'], 4),
        'trajectory_pattern': traj['pattern'],
        'skill_score': round(ss, 4),
        'behavioral_score': round(bs, 4),
        'ai_focus_ratio': round(traj['ai_focus_ratio'], 4),
    }

print('Scoring module ready')

In [ ]:
# ── Reasoning ──────────────────────────────────────────────────────
def reasoning(c, sc):
    p = c.get('profile',{})
    sig = c.get('redrob_signals',{})
    skills = [s['name'] for s in sorted(c.get('skills',[]),
              key=lambda x: {'expert':3,'advanced':2,'intermediate':1,'beginner':0}.get(x.get('proficiency','beginner'),0),
              reverse=True)][:4]
    notice = sig.get('notice_period_days',90)
    avail = 'immediately available' if notice==0 and sig.get('open_to_work_flag') else f'{notice}d notice'
    s1 = f"{p.get('current_title','N/A')} at {p.get('current_company','N/A')} with {p.get('years_of_experience',0)}yr exp; skills: {', '.join(skills)}."
    s2 = f"Career trajectory: {sc['trajectory_pattern']} (score {sc['trajectory_score']:.2f}), {sc['ai_focus_ratio']*100:.0f}% of career in AI/ML."
    s3 = f"Availability: {avail}; recruiter response rate {sig.get('recruiter_response_rate',0)*100:.0f}%."
    return f'{s1} {s2} {s3}'

print('Reasoning module ready')

In [ ]:
# ── Score + Rank + Output ──────────────────────────────────────────
print('Scoring candidates...')
scored = []
for cid, sem in tqdm(top_results):
    c = candidate_map.get(cid)
    if c:
        sc = hybrid_score(sem, c)
        scored.append((cid, sc))

scored.sort(key=lambda x: x[1]['final_score'], reverse=True)
top100 = scored[:TOP_N]

print('Generating reasoning...')
results = []
for rank, (cid, sc) in enumerate(top100, 1):
    c = candidate_map[cid]
    results.append({
        'rank': rank,
        'candidate_id': cid,
        'final_score': sc['final_score'],
        'semantic_score': sc['semantic_score'],
        'trajectory_score': sc['trajectory_score'],
        'trajectory_pattern': sc['trajectory_pattern'],
        'skill_score': sc['skill_score'],
        'behavioral_score': sc['behavioral_score'],
        'reasoning': reasoning(c, sc)
    })

# Write CSV
fields = ['rank','candidate_id','final_score','semantic_score','trajectory_score',
          'trajectory_pattern','skill_score','behavioral_score','reasoning']
with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(results)

print(f'\n✅ Done! submission.csv written with {len(results)} candidates')
print('\nTop 5:')
for r in results[:5]:
    print(f"  #{r['rank']} {r['candidate_id']} | Score: {r['final_score']:.4f} | {r['trajectory_pattern']}")

In [ ]:
# ── Download submission.csv ────────────────────────────────────────
from google.colab import files
files.download('submission.csv')
print('Downloaded!')